# Chapter 11: Further Issues in Using OLS with Time Series Data

Topics: asymptotics, highly persistent time series, unit roots, random walks, first differences.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 11.2 Simulating Random Walks

## Step 2 – Stationary vs. Integrated (Unit Root) Processes

A process $\{y_t\}$ is **covariance stationary** if $E(y_t)$, $\text{Var}(y_t)$, and $\text{Cov}(y_t, y_{t-h})$ are all time-invariant.

**Random walk (RW)** — simplest I(1) process:
$$y_t = y_{t-1} + e_t, \quad e_t \sim i.i.d.(0,\sigma^2)$$

- $E(y_t) = y_0$, $\text{Var}(y_t) = t\sigma^2$ → grows with $t$ (non-stationary)

**Random walk with drift:**
$$y_t = \alpha + y_{t-1} + e_t$$

- $E(y_t) = y_0 + \alpha t$ — exhibits a linear trend

**AR(1) process:**
$$y_t = \rho y_{t-1} + e_t$$

- Stationary if $|\rho| < 1$; has a unit root (I(1)) if $\rho = 1$

In [ ]:
np.random.seed(42)
T = 100
n_paths = 5

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for _ in range(n_paths):
    rw = np.cumsum(np.random.normal(0, 1, T))
    axes[0].plot(rw, alpha=0.7)
axes[0].set_title('Random Walks'); axes[0].set_xlabel('t')

# With drift
for _ in range(n_paths):
    rw_drift = np.cumsum(0.1 + np.random.normal(0, 1, T))
    axes[1].plot(rw_drift, alpha=0.7)
axes[1].set_title('Random Walks with Drift (μ=0.1)'); axes[1].set_xlabel('t')

plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_random_walks.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_11_Persistent_Time_Series_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11.2 Unit Root Tests (ADF Test)

## Step 3 – Augmented Dickey-Fuller (ADF) Test

The ADF tests H₀: unit root ($\gamma = 0$) vs. H₁: stationarity ($\gamma < 0$).

**Test regression:**
$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{j=1}^p \phi_j \Delta y_{t-j} + e_t$$

**ADF statistic:** $\tau = \hat{\gamma}/\text{se}(\hat{\gamma})$

⚠️ Critical values come from the **Dickey-Fuller distribution** (not standard t). These are more negative than standard t critical values.

**First differencing** to achieve stationarity for an I(1) series:
$$\Delta y_t = y_t - y_{t-1}$$

An I(1) series becomes I(0) after one difference.

In [ ]:
df = woo.dataWoo('phillips')

for var in ['inf', 'unem']:
    adf = adfuller(df[var].dropna(), autolag='AIC')
    print(f"\nADF Test for '{var}':")
    print(f"  ADF Statistic: {adf[0]:.4f}")
    print(f"  p-value:       {adf[1]:.4f}")
    print(f"  Lags used:     {adf[2]}")
    print(f"  Critical values: {adf[4]}")
    conclusion = "Non-stationary (unit root)" if adf[1] > 0.05 else "Stationary"
    print(f"  Conclusion: {conclusion}")

## 11.4 First Differences

In [ ]:
df['d_inf']  = df['inf'].diff()
df['d_unem'] = df['unem'].diff()

# Check stationarity of differences
adf_dinf = adfuller(df['d_inf'].dropna())
print(f"ADF on Δinf:  stat={adf_dinf[0]:.4f}, p={adf_dinf[1]:.4f}")

m_diff = smf.ols('d_inf ~ d_unem', data=df.dropna()).fit()
print("\nRegression in first differences:")
print(m_diff.summary())